# `llama-server` on Kaggle — full notebook

This notebook is a full Kaggle workflow for launching a modern `llama-server`
runtime on Kaggle notebooks, packaging or publishing build artifacts,
and validating text or multimodal inference through the local OpenAI-compatible
server surface.

Why this notebook exists:

- older `llama.cpp` builds that were good enough for historical models are not
  sufficient for newer model families
- in particular, newer Gemma 4 era models need a newer working runtime path
- this notebook is intended to make that runtime reproducible on Kaggle VMs
  and adaptable to the supported accelerator profile of the current session

Public companion assets:

- CUDA runtime build dataset for the currently validated Kaggle CUDA path:
  `https://www.kaggle.com/datasets/alexandreev/llama-server-cuda-build-kaggle`
- ready-to-load GGUF model family:
  `https://www.kaggle.com/models/alexandreev/gemma-4-26b-a4b-it-gguf`

More model artifacts may be added later. Stay tuned.

This notebook is the full workflow:

- guarded against accidental auto-execution
- environment checks
- artifact discovery and unpacking
- optional rebuild from source
- validation of the binary and runtime libraries
- optional packaging and Kaggle dataset upload or update
- isolated server sandbox
- start / stop / restart helpers
- text and multimodal client helpers
- structured artifact export at the end of useful runs
- workflow-oriented execution with lightweight guards for risky stages

It is designed to be safe by default:

- the first code cell aborts immediately until you opt in
- the runtime is copied into a sandbox before launch
- no flag with a missing value is emitted
- media support is disabled unless explicitly enabled

It also supports two explicit runtime profiles:

- `gpu` — CUDA build, GPU monitoring, Kaggle dataset slug for the CUDA artifact
- `cpu` — CPU-only build, RAM-focused monitoring, separate Kaggle dataset slug for the CPU artifact

Keep GPU and CPU artifacts in different Kaggle datasets.


## Recommended workflows

Choose one `SESSION_WORKFLOW` before enabling execution. A workflow may include several naturally consecutive actions, such as build, validation, and publish, but the notebook should still have one clear execution path.

- `runtime_demo`: processes `10 -> 20 -> 30 -> 60 -> 70 -> 80 -> 100`
- `build_validate`: processes `10 -> 20 -> 30 -> 40 -> 60 -> 70 -> 100`
- `build_validate_publish`: processes `10 -> 20 -> 30 -> 40 -> 50 -> 60 -> 70 -> 100`
- `text_demo`: processes `10 -> 20 -> 30 -> 60 -> 70 -> 80 -> 100`
- `multimodal_demo`: processes `10 -> 20 -> 30 -> 60 -> 70 -> 90 -> 100`
- `profiling`: processes `10 -> 20 -> 30 -> 60 -> 70 -> 80 -> 100`
- `project_handoff`: processes `10 -> 20 -> 30 -> 60 -> 70 -> 100`

The numbers are intentionally visible. They make it easier to scan the notebook from top to bottom and understand which sections belong to the current workflow without repeatedly looking up helper state.


## Release notes

- `v1`: Replaced strict single-goal guidance with a simpler `SESSION_WORKFLOW` model for natural multi-step Kaggle sessions.
- Added numbered process mapping so the execution order is visible directly in the notebook body.
- Added guarded compile, publish, demo, and artifact-export paths to reduce accidental execution.
- Added structured artifact export for `run_manifest.json`, `profile_report.json`, and `llama-server.log` handoff.
- Clarified the notebook intro, neutralized notebook naming, and documented the current public build and model links.


In [ ]:
# RUN GUARD
RUN_NOTEBOOK = False
SESSION_WORKFLOW = "runtime_demo"
NOTEBOOK_VERSION = "v1"
LAST_UPDATED = "2026-04-06"

PROCESS_DESCRIPTIONS = {
    10: "Environment checks",
    20: "Artifact discovery and unpacking",
    30: "Model discovery",
    40: "Optional rebuild from source",
    50: "Packaging and Kaggle dataset upload",
    60: "Isolated runtime sandbox and server management",
    70: "Health, models, and slots",
    80: "Text helpers and text demos",
    90: "Multimodal helpers",
    100: "Export run artifacts",
}

WORKFLOW_PROCESSES = {
    "runtime_demo": (10, 20, 30, 60, 70, 80, 100),
    "build_validate": (10, 20, 30, 40, 60, 70, 100),
    "build_validate_publish": (10, 20, 30, 40, 50, 60, 70, 100),
    "text_demo": (10, 20, 30, 60, 70, 80, 100),
    "multimodal_demo": (10, 20, 30, 60, 70, 90, 100),
    "profiling": (10, 20, 30, 60, 70, 80, 100),
    "project_handoff": (10, 20, 30, 60, 70, 100),
}

if not RUN_NOTEBOOK:
    raise RuntimeError(
        "Notebook execution is disabled. Set RUN_NOTEBOOK = True only after reading the intro, choosing a workflow, and confirming the attached artifacts."
    )

if SESSION_WORKFLOW not in WORKFLOW_PROCESSES:
    raise ValueError(f"Unsupported SESSION_WORKFLOW: {SESSION_WORKFLOW!r}")

print(f"Notebook version: {NOTEBOOK_VERSION} ({LAST_UPDATED})")
print(f"SESSION_WORKFLOW: {SESSION_WORKFLOW}")
print("ENABLED PROCESSES:")
for process_id in WORKFLOW_PROCESSES[SESSION_WORKFLOW]:
    print(f" - {process_id}: {PROCESS_DESCRIPTIONS[process_id]}")


## Operator inputs

Set the required runtime inputs here before running any numbered process.

These values are intentionally explicit and near the top of the notebook:
- `SERVER_PROFILE` selects the runtime artifact family
- `GPU_ARTIFACT_DATASET_ROOT` is the full Kaggle input path to the GPU server artifact
- `CPU_ARTIFACT_DATASET_ROOT` is the full Kaggle input path to the CPU server artifact
- `MODEL_FILE` must be the full path to the attached GGUF file
- `MMPROJ_FILE` must be the full path to the projection file when the chosen model needs one

This notebook does not search for artifacts or model files on your behalf.


In [ ]:
# REQUIRED OPERATOR INPUTS
SERVER_PROFILE = "gpu"
GPU_ARTIFACT_DATASET_ROOT = "/kaggle/input/path/to/your/gpu-runtime-artifact"
CPU_ARTIFACT_DATASET_ROOT = "/kaggle/input/path/to/your/cpu-runtime-artifact"
MODEL_FILE = "/kaggle/input/path/to/your/model.gguf"
MMPROJ_FILE = ""  # Set a full path when the selected model requires a projection file.

# OPTIONAL OPERATIONAL OVERRIDES
# Set any of these only when you intentionally want to override notebook defaults.
# WORK_ROOT = "/kaggle/working/llama_server"
# HOST = "127.0.0.1"
# PORT = 18081
# CTX_SIZE = 2048
# BATCH_SIZE = 2048
# UBATCH_SIZE = 512
# PARALLEL = 1
# SPLIT_MODE = None
# TENSOR_SPLIT = None
# SLOTS_ENDPOINT = None
# EXTRA_SERVER_ARGS = []
# DO_COMPILE = False
# DATASET_MODE = "skip"
# VERSION_NOTE = "Update runtime artifact"
# MAKE_PUBLIC = False
# STARTUP_TIMEOUT_SECONDS = 120
# CLIENT_TIMEOUT_SECONDS = 120
# MAX_STABLE_CONTEXT_OBSERVED = 130000


In [ ]:
# These helpers are intentionally grouped together because they define the notebook's
# single policy layer for workflow-aware execution. Every later guarded cell should
# depend on this block instead of re-implementing workflow checks ad hoc.
#
# The same cell also hosts the shared foundation imports and configuration because
# every numbered process depends on them. Keeping that state in one explicit place
# prevents hidden initialization order bugs across the notebook.
#
# Remaining globals().get(...) calls are only for optional operational overrides.
# They let an operator tune runtime behavior in an earlier cell without editing the
# shared foundation block itself. Required inputs such as SERVER_PROFILE, artifact
# roots, MODEL_FILE, and MMPROJ_FILE now live in the dedicated cell above.

from pathlib import Path
import base64
import hashlib
import json
import mimetypes
import os
import shutil
import signal
import socket
import subprocess
import time
import zipfile
from typing import Any


def workflow_processes(workflow: str | None = None) -> tuple[int, ...]:
    """Return the ordered process ids enabled for the active or requested workflow."""
    if workflow is None:
        workflow = SESSION_WORKFLOW
    return WORKFLOW_PROCESSES[workflow]


def process_enabled(process_id: int, workflow: str | None = None) -> bool:
    """Return True when a process id belongs to the active or requested workflow."""
    return process_id in workflow_processes(workflow)


def require_process(process_id: int) -> None:
    """Raise a clear error when a guarded cell is used outside its workflow."""
    if process_enabled(process_id):
        return
    enabled = workflow_processes()
    raise RuntimeError(
        f"Process {process_id} is not enabled for SESSION_WORKFLOW={SESSION_WORKFLOW!r}. Enabled processes: {enabled}"
    )


def guarded_toggle(flag: bool, *, process_id: int, label: str) -> bool:
    """Enable an optional cell only when both the flag and workflow process allow it."""
    if not flag:
        return False
    require_process(process_id)
    print(f"{label}: enabled for process {process_id} in SESSION_WORKFLOW={SESSION_WORKFLOW}")
    return True


PROFILE_SETTINGS = globals().get("PROFILE_SETTINGS", {
    "gpu": {
        "artifact_dataset_root": Path(GPU_ARTIFACT_DATASET_ROOT),
        "dataset_slug": "llama-server-cuda-build-kaggle",
        "dataset_title": "llama-server CUDA build Kaggle",
        "dataset_subtitle": "Prebuilt llama.cpp server binary with CUDA runtime libs",
        "n_gpu_layers": "all",
        "flash_attn": "on",
        "use_cuda_runtime": True,
        "monitor_kind": "gpu",
        "startup_timeout_seconds": 120,
        "client_timeout_seconds": 120,
    },
    "cpu": {
        "artifact_dataset_root": Path(CPU_ARTIFACT_DATASET_ROOT),
        "dataset_slug": "llama-server-cpu-build-kaggle",
        "dataset_title": "llama-server CPU build Kaggle",
        "dataset_subtitle": "Prebuilt llama.cpp server binary without CUDA runtime libs",
        "n_gpu_layers": 0,
        "flash_attn": "off",
        "use_cuda_runtime": False,
        "monitor_kind": "ram",
        "startup_timeout_seconds": 300,
        "client_timeout_seconds": 300,
    },
})
if SERVER_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f"Unsupported SERVER_PROFILE: {SERVER_PROFILE!r}")
PROFILE = PROFILE_SETTINGS[SERVER_PROFILE]

ARTIFACT_DATASET_ROOT = PROFILE["artifact_dataset_root"]
MODEL_FILE = Path(MODEL_FILE)
if not str(MODEL_FILE):
    raise ValueError("MODEL_FILE must be set to the full path of the attached GGUF model file before running the notebook.")
if not MODEL_FILE.exists() or not MODEL_FILE.is_file():
    raise FileNotFoundError(f"MODEL_FILE does not exist: {MODEL_FILE}")

MMPROJ_FILE = Path(MMPROJ_FILE) if MMPROJ_FILE else None
if MMPROJ_FILE is not None and (not MMPROJ_FILE.exists() or not MMPROJ_FILE.is_file()):
    raise FileNotFoundError(f"MMPROJ_FILE does not exist: {MMPROJ_FILE}")


WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working/llama_server"))
PREBUILT_ROOT = globals().get("PREBUILT_ROOT", WORK_ROOT / "prebuilt")
SRC_ROOT = globals().get("SRC_ROOT", WORK_ROOT / "src")
PACKAGE_ROOT = globals().get("PACKAGE_ROOT", WORK_ROOT / "package")
SANDBOX_BASE = globals().get("SANDBOX_BASE", WORK_ROOT / "sandbox")
LOG_DIR = globals().get("LOG_DIR", WORK_ROOT / "logs")
RUN_DIR = globals().get("RUN_DIR", WORK_ROOT / "run")
for root in [WORK_ROOT, PREBUILT_ROOT, SRC_ROOT, PACKAGE_ROOT, SANDBOX_BASE, LOG_DIR, RUN_DIR]:
    root.mkdir(parents=True, exist_ok=True)

HOST = globals().get("HOST", "127.0.0.1")
PORT = globals().get("PORT", 18081)
CTX_SIZE = globals().get("CTX_SIZE", 2048)
BATCH_SIZE = globals().get("BATCH_SIZE", CTX_SIZE)
UBATCH_SIZE = globals().get("UBATCH_SIZE", 512)
PARALLEL = globals().get("PARALLEL", 1)
N_GPU_LAYERS = PROFILE["n_gpu_layers"]
FLASH_ATTN = PROFILE["flash_attn"]
USE_CUDA_RUNTIME = PROFILE["use_cuda_runtime"]
MONITOR_KIND = PROFILE["monitor_kind"]
STARTUP_TIMEOUT_SECONDS = globals().get("STARTUP_TIMEOUT_SECONDS", PROFILE["startup_timeout_seconds"])
CLIENT_TIMEOUT_SECONDS = globals().get("CLIENT_TIMEOUT_SECONDS", PROFILE["client_timeout_seconds"])
MAX_STABLE_CONTEXT_OBSERVED = globals().get("MAX_STABLE_CONTEXT_OBSERVED", None)
SPLIT_MODE = globals().get("SPLIT_MODE", None)
TENSOR_SPLIT = globals().get("TENSOR_SPLIT", None)
SLOTS_ENDPOINT = globals().get("SLOTS_ENDPOINT", None)
MEDIA_ROOT = globals().get("MEDIA_ROOT", None)
EXTRA_SERVER_ARGS = globals().get("EXTRA_SERVER_ARGS", [
    "--cache-type-k", "q4_0",
    "--cache-type-v", "q4_0",
    "--reasoning", "off",
    "--reasoning-budget", "0",
    "--reasoning-format", "none",
])
DATASET_USERNAME = globals().get("DATASET_USERNAME", "alexandreev")
DATASET_SLUG = PROFILE["dataset_slug"]
DATASET_TITLE = PROFILE["dataset_title"]
DATASET_SUBTITLE = PROFILE["dataset_subtitle"]
CUDA_ROOT = globals().get("CUDA_ROOT", "/usr/local/cuda-12.8")
CUDA_DRIVER_LIB = globals().get("CUDA_DRIVER_LIB", "/usr/local/nvidia/lib64/libcuda.so")
PID_FILE = globals().get("PID_FILE", RUN_DIR / "llama-server.pid")
CMD_FILE = globals().get("CMD_FILE", RUN_DIR / "llama-server.command.json")
LOG_FILE = globals().get("LOG_FILE", LOG_DIR / "llama-server.log")
GIT_REF = globals().get("GIT_REF", "master")
DO_COMPILE = globals().get("DO_COMPILE", False)
PACKAGE_STYLE = globals().get("PACKAGE_STYLE", "flat")
DATASET_MODE = globals().get("DATASET_MODE", "skip")
VERSION_NOTE = globals().get("VERSION_NOTE", "Update runtime artifact")
MAKE_PUBLIC = globals().get("MAKE_PUBLIC", False)


def _copy_runtime_layout(dataset_root: Path, out_root: Path) -> str:
    """Copy a supported runtime artifact layout from an attached dataset into working storage."""
    if (dataset_root / "llama-server").exists():
        for item in dataset_root.iterdir():
            if item.is_file():
                shutil.copy2(item, out_root / item.name)
        return "flat"

    if (dataset_root / "bin" / "llama-server").exists():
        for item in (dataset_root / "bin").iterdir():
            if item.is_file():
                shutil.copy2(item, out_root / item.name)
        return "bin_dir"

    if (dataset_root / "runtime" / "llama-server").exists():
        for item in (dataset_root / "runtime").iterdir():
            if item.is_file():
                shutil.copy2(item, out_root / item.name)
        return "runtime_dir"

    zip_files = sorted(dataset_root.glob("*.zip"))
    if zip_files:
        with zipfile.ZipFile(zip_files[0], "r") as archive:
            archive.extractall(out_root)
        extracted_runtime = out_root / "runtime"
        if not (extracted_runtime / "llama-server").exists():
            return "zip"
        tmp_root = out_root / "__tmp__"
        tmp_root.mkdir(parents=True, exist_ok=True)
        for item in extracted_runtime.iterdir():
            if item.is_file():
                shutil.move(str(item), str(tmp_root / item.name))
        shutil.rmtree(out_root)
        out_root.mkdir(parents=True, exist_ok=True)
        for item in tmp_root.iterdir():
            shutil.move(str(item), str(out_root / item.name))
        shutil.rmtree(tmp_root, ignore_errors=True)
        return "zip"

    raise FileNotFoundError(
        f"No supported runtime layout found under {dataset_root}. For SERVER_PROFILE={SERVER_PROFILE!r}, attach the matching Kaggle dataset."
    )


def prepare_runtime_from_dataset(dataset_root: Path, out_root: Path, force_restage: bool = False) -> dict[str, Any]:
    """Stage a runnable server artifact into the notebook working directory."""
    if (out_root / "llama-server").exists() and not force_restage:
        return {"layout": "working_copy", "runtime_root": str(out_root)}
    if out_root.exists():
        shutil.rmtree(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    layout = _copy_runtime_layout(dataset_root, out_root)
    return {"layout": layout, "runtime_root": str(out_root)}


## Process 10 — Environment checks

This section gathers only low-risk host facts needed before touching artifacts or launching a server. The helper and the check list stay together because they form one small diagnostic unit: define a shell probe, run it, and print its result in a uniform format.


In [ ]:
# Process 10 code path.
require_process(10)

def run_check(name: str, cmd: str) -> dict[str, str]:
    """Execute one shell probe and return a normalized printable record."""
    proc = subprocess.run(["bash", "-lc", cmd], text=True, capture_output=True)
    return {
        "name": name,
        "command": cmd,
        "returncode": str(proc.returncode),
        "stdout": proc.stdout.strip(),
        "stderr": proc.stderr.strip(),
    }


checks = [
    ("profile", f"printf '%s\n' '{SERVER_PROFILE}'"),
    ("cmake", "cmake --version | head -1"),
    ("git", "git --version"),
    ("kaggle", "kaggle --version || true"),
    ("disk", "df -h /kaggle/working || true"),
]
if USE_CUDA_RUNTIME:
    checks += [
        ("nvcc", "which nvcc && nvcc --version || true"),
        ("nvidia_smi", "nvidia-smi || true"),
        ("cuda_driver", "ls -l /usr/local/nvidia/lib64/libcuda.so* || true"),
    ]
else:
    checks += [
        ("lscpu", "lscpu | sed -n '1,20p' || true"),
        ("ram", "free -h || true"),
    ]

for item in [run_check(name, cmd) for name, cmd in checks]:
    print("=" * 80)
    print(item["name"])
    print("cmd:", item["command"])
    print("returncode:", item["returncode"])
    if item["stdout"]:
        print("-- stdout --")
        print(item["stdout"])
    if item["stderr"]:
        print("-- stderr --")
        print(item["stderr"])


## Process 20 — Artifact discovery and unpacking

This section inspects attached Kaggle datasets and local staging roots. The helper is intentionally small and local to this process: it only exists to make artifact trees readable while confirming that the expected build bundle is present.


In [ ]:
# Process 20 code path.
require_process(20)

def list_tree(root: Path, max_depth: int = 2) -> list[str]:
    """Return a shallow printable tree for quick inspection of attached artifacts."""
    out = []
    if not root.exists():
        return out
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        if len(rel.parts) > max_depth:
            continue
        out.append(str(rel) + ("/" if p.is_dir() else ""))
    return out


for item in list_tree(ARTIFACT_DATASET_ROOT, max_depth=2)[:200]:
    print(" -", item)


In [ ]:
# Process 20 code path.
require_process(20)

# BOOTSTRAP CELL
# After a Kaggle VM reboot, rerun this cell first. It restores the staged runtime
# artifact under the notebook's own working directory without rebuilding the server.


def bootstrap_runtime(force_restage: bool = False) -> dict[str, Any]:
    """Stage the selected runtime artifact and expose the prebuilt server paths."""
    global PREBUILT_BINARY, PREBUILT_WRAPPER, ACTIVE_BINARY, STAGED_RUNTIME_ROOT

    prepared = prepare_runtime_from_dataset(ARTIFACT_DATASET_ROOT, PREBUILT_ROOT, force_restage=force_restage)
    PREBUILT_BINARY = PREBUILT_ROOT / 'llama-server'
    PREBUILT_WRAPPER = PREBUILT_ROOT / 'llama-server.sh'
    if not PREBUILT_BINARY.exists():
        raise SystemExit('Prepared runtime does not contain llama-server')

    for candidate in [PREBUILT_BINARY, PREBUILT_WRAPPER, PREBUILT_ROOT / 'llama-cli', PREBUILT_ROOT / 'llama-cli.sh']:
        if candidate.exists():
            candidate.chmod(candidate.stat().st_mode | 0o111)

    ACTIVE_BINARY = PREBUILT_BINARY
    STAGED_RUNTIME_ROOT = PREBUILT_ROOT
    return {
        'server_profile': SERVER_PROFILE,
        'artifact_dataset_root': str(ARTIFACT_DATASET_ROOT),
        'prebuilt_root': str(PREBUILT_ROOT),
        'active_binary': str(ACTIVE_BINARY),
        'runtime_source': prepared['layout'],
        'monitor_kind': MONITOR_KIND,
    }


bootstrap_info = bootstrap_runtime(force_restage=False)
print(json.dumps(bootstrap_info, indent=2, ensure_ascii=False))
print('Runtime files:')
for item in sorted(PREBUILT_ROOT.iterdir()):
    if item.is_file():
        print(' -', item.name)


In [ ]:
# Process 20 code path.
require_process(20)

# This validation step confirms that the staged artifact can report a version and
# that the current accelerator profile is visible before the notebook moves on.


def staged_runtime_env() -> dict[str, str]:
    """Return the environment used for quick checks against the staged runtime."""
    env = os.environ.copy()
    ld_parts = [str(PREBUILT_ROOT)]
    if USE_CUDA_RUNTIME:
        for part in [
            '/usr/local/nvidia/lib64',
            '/usr/local/cuda/lib64',
            '/usr/local/cuda-12.8/lib64',
            '/usr/local/cuda-12.8/targets/x86_64-linux/lib',
        ]:
            if Path(part).exists():
                ld_parts.append(part)
    if env.get('LD_LIBRARY_PATH'):
        ld_parts.append(env['LD_LIBRARY_PATH'])
    env['LD_LIBRARY_PATH'] = ':'.join(ld_parts).strip(':')
    return env


def staged_runtime_command() -> Path:
    """Return the executable used for quick validation of the staged runtime."""
    if PREBUILT_WRAPPER.exists():
        return PREBUILT_WRAPPER
    return PREBUILT_BINARY


version_proc = subprocess.run([str(staged_runtime_command()), '--version'], text=True, capture_output=True, env=staged_runtime_env())
print(version_proc.stdout or version_proc.stderr)

if USE_CUDA_RUNTIME:
    probe_proc = subprocess.run(['bash', '-lc', 'nvidia-smi || true'], text=True, capture_output=True)
    print('==== accelerator ====')
    print(probe_proc.stdout or probe_proc.stderr)
else:
    probe_proc = subprocess.run(['bash', '-lc', 'free -h || true'], text=True, capture_output=True)
    print('==== ram ====')
    print(probe_proc.stdout or probe_proc.stderr)


## Process 30 — Model binding

This section binds the notebook to explicitly chosen model files. File selection is not part of the notebook contract: attach whatever Kaggle model artifact you want and pass its full path through `MODEL_FILE`. If the model family also needs a projection file, pass its full path through `MMPROJ_FILE`. The notebook does not search for either file.


In [ ]:
# Process 30 code path.
require_process(30)

# The notebook does not search for model files. The operator must provide the exact
# GGUF file path through MODEL_FILE and, when needed, the exact projection path
# through MMPROJ_FILE before execution.

print('MODEL_FILE =', MODEL_FILE)
print('MMPROJ_FILE =', MMPROJ_FILE)


## Process 40 — Optional rebuild from source

This section is only for workflows that intentionally compile or rebuild `llama.cpp`. It stays isolated because build logic is expensive, environment-sensitive, and should not run accidentally during ordinary runtime or demo sessions.


In [ ]:
# Process 40 code path.
require_process(40)

# This cell keeps build-specific logic local to process 40.
# The small selector functions avoid a long top-level if/else ladder in the cell body,
# so the execution flow stays easy to scan.

RUN_COMPILE = DO_COMPILE


def cuda_build_script() -> str:
    """Return the CUDA-enabled build script for the current Kaggle runtime."""
    return f"""
    set -euxo pipefail
    rm -rf "{SRC_ROOT}"
    git clone --depth 1 --branch "{GIT_REF}" https://github.com/ggml-org/llama.cpp.git "{SRC_ROOT}"
    cd "{SRC_ROOT}"

    export PATH="{CUDA_ROOT}/bin:$PATH"
    export LD_LIBRARY_PATH="/usr/local/nvidia/lib64:{CUDA_ROOT}/lib64:{CUDA_ROOT}/targets/x86_64-linux/lib:${{LD_LIBRARY_PATH:-}}"
    export LIBRARY_PATH="/usr/local/nvidia/lib64:{CUDA_ROOT}/lib64:{CUDA_ROOT}/targets/x86_64-linux/lib:/usr/local/cuda/lib64/stubs:${{LIBRARY_PATH:-}}"

    cmake -S . -B build \
      -DGGML_CUDA=ON \
      -DGGML_NATIVE=OFF \
      -DCMAKE_BUILD_TYPE=Release \
      -DCUDAToolkit_ROOT={CUDA_ROOT} \
      -DCMAKE_CUDA_COMPILER={CUDA_ROOT}/bin/nvcc \
      -DCMAKE_CUDA_ARCHITECTURES=75 \
      -DCMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64 \
      -DCUDA_cuda_driver_LIBRARY={CUDA_DRIVER_LIB}

    cmake --build build --config Release -j 4 --target llama-server llama-cli
    ./build/bin/llama-server --version
    """


def cpu_build_script() -> str:
    """Return the CPU-only build script for the current Kaggle runtime."""
    return f"""
    set -euxo pipefail
    rm -rf "{SRC_ROOT}"
    git clone --depth 1 --branch "{GIT_REF}" https://github.com/ggml-org/llama.cpp.git "{SRC_ROOT}"
    cd "{SRC_ROOT}"

    cmake -S . -B build \
      -DGGML_CUDA=OFF \
      -DGGML_NATIVE=OFF \
      -DCMAKE_BUILD_TYPE=Release

    cmake --build build --config Release -j 4 --target llama-server llama-cli
    ./build/bin/llama-server --version
    """


def selected_build_script() -> str:
    """Return the build script matching the active server profile."""
    if USE_CUDA_RUNTIME:
        return cuda_build_script()
    return cpu_build_script()


if not guarded_toggle(RUN_COMPILE, process_id=40, label="Compile from source"):
    print("Skipping source build")
else:
    proc = subprocess.run(["bash", "-lc", selected_build_script()], text=True, capture_output=True)
    print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"Build failed with exit code {proc.returncode}")


In [ ]:
# Process 40 code path.
require_process(40)

# This bridge cell resolves which binary should be treated as active after process 40.
# It keeps the choice explicit instead of hiding it inside later packaging or runtime cells.


def resolve_active_binary(compiled_binary: Path, prebuilt_binary: Path) -> Path:
    """Prefer a freshly compiled server binary, otherwise fall back to the prebuilt artifact."""
    if compiled_binary.exists():
        return compiled_binary
    return prebuilt_binary


COMPILED_BINARY = SRC_ROOT / "build" / "bin" / "llama-server"
ACTIVE_BINARY = resolve_active_binary(COMPILED_BINARY, PREBUILT_BINARY)
print("ACTIVE_BINARY =", ACTIVE_BINARY)


## Process 50 — Packaging and Kaggle dataset upload

This section turns a validated runtime into a reusable artifact and optionally versions it as a Kaggle dataset. Packaging and publish helpers stay together because they belong to the same handoff boundary: prepare a clean bundle, then send it to Kaggle when explicitly requested.


In [ ]:
# Process 50 code path.
require_process(50)

# These helpers stay in one cell because packaging is a single handoff concern:
# gather runtime files, preserve metadata, and emit one reproducible bundle.

def copy_real(src: Path, dst: Path):
    """Copy a real file and preserve its executable mode for packaged runtimes."""
    shutil.copy2(src.resolve(), dst)
    dst.chmod(src.resolve().stat().st_mode)

def package_runtime_bundle(active_binary: Path, active_root: Path, out_root: Path, style: str = "flat", use_cuda_runtime: bool | None = None) -> Path:
    """Assemble a self-contained runtime bundle and capture reproducibility metadata."""
    use_cuda_runtime = USE_CUDA_RUNTIME if use_cuda_runtime is None else use_cuda_runtime
    if out_root.exists():
        shutil.rmtree(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    runtime_target = out_root if style == "flat" else (out_root / "runtime")
    runtime_target.mkdir(parents=True, exist_ok=True)

    copy_real(active_binary, runtime_target / "llama-server")
    for name in ["llama-cli", "llama-server.sh", "llama-cli.sh"]:
        src = active_root / name
        if src.exists():
            copy_real(src, runtime_target / name)
    for src in sorted(active_root.glob("*.so*")):
        if src.is_file():
            copy_real(src, runtime_target / src.name)

    wrapper = runtime_target / "llama-server.sh"
    if not wrapper.exists():
        wrapper.write_text(
            '#!/usr/bin/env bash\n'
            'set -euo pipefail\n'
            'DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"\n'
            'export LD_LIBRARY_PATH="$DIR:${LD_LIBRARY_PATH:-}"\n'
            'exec "$DIR/llama-server" "$@"\n',
            encoding='utf-8',
        )
        wrapper.chmod(0o755)

    env = os.environ.copy()
    ld_parts = [str(runtime_target)]
    if use_cuda_runtime:
        for part in [
            "/usr/local/nvidia/lib64",
            "/usr/local/cuda/lib64",
            "/usr/local/cuda-12.8/lib64",
            "/usr/local/cuda-12.8/targets/x86_64-linux/lib",
        ]:
            if Path(part).exists():
                ld_parts.append(part)
    if env.get("LD_LIBRARY_PATH"):
        ld_parts.append(env["LD_LIBRARY_PATH"])
    env["LD_LIBRARY_PATH"] = ":".join(ld_parts).strip(":")

    proc = subprocess.run([str(wrapper), "--version"], text=True, capture_output=True, env=env)
    (out_root / "llama-server.version.txt").write_text((proc.stdout or proc.stderr), encoding='utf-8')
    ldd_before = subprocess.run(["ldd", str(active_binary)], text=True, capture_output=True)
    (out_root / "llama-server.ldd.before.txt").write_text((ldd_before.stdout or ldd_before.stderr), encoding='utf-8')
    ldd_after = subprocess.run(["ldd", str(runtime_target / "llama-server")], text=True, capture_output=True, env=env)
    (out_root / "llama-server.ldd.after.txt").write_text((ldd_after.stdout or ldd_after.stderr), encoding='utf-8')

    if use_cuda_runtime:
        nvcc_out = subprocess.run(["bash", "-lc", "nvcc --version || true"], text=True, capture_output=True)
        (out_root / "nvcc.version.txt").write_text(nvcc_out.stdout or nvcc_out.stderr, encoding='utf-8')
        smi_out = subprocess.run(["bash", "-lc", "nvidia-smi || true"], text=True, capture_output=True)
        (out_root / "nvidia-smi.txt").write_text(smi_out.stdout or smi_out.stderr, encoding='utf-8')
    else:
        ram_out = subprocess.run(["bash", "-lc", "free -h || true"], text=True, capture_output=True)
        (out_root / "ram.usage.txt").write_text(ram_out.stdout or ram_out.stderr, encoding='utf-8')
        cpu_out = subprocess.run(["bash", "-lc", "lscpu || true"], text=True, capture_output=True)
        (out_root / "cpu.info.txt").write_text(cpu_out.stdout or cpu_out.stderr, encoding='utf-8')

    if (SRC_ROOT / ".git").exists():
        git_out = subprocess.run(["git", "-C", str(SRC_ROOT), "rev-parse", "HEAD"], text=True, capture_output=True)
        (out_root / "llama.cpp.commit.txt").write_text((git_out.stdout or git_out.stderr), encoding='utf-8')
    if (SRC_ROOT / "LICENSE").exists():
        shutil.copy2(SRC_ROOT / "LICENSE", out_root / "LICENSE.llama.cpp.txt")

    hashes = []
    for p in sorted(runtime_target.iterdir()):
        if p.is_file():
            h = hashlib.sha256()
            with open(p, 'rb') as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b''):
                    h.update(chunk)
            hashes.append(f"{h.hexdigest()}  {p.relative_to(out_root).as_posix()}")
    (out_root / "sha256sums.txt").write_text("\n".join(hashes) + ("\n" if hashes else ""), encoding='utf-8')

    profile_info = {
        "server_profile": SERVER_PROFILE,
        "dataset_slug": DATASET_SLUG,
        "use_cuda_runtime": bool(use_cuda_runtime),
        "monitor_kind": MONITOR_KIND,
    }
    (out_root / "runtime-profile.json").write_text(json.dumps(profile_info, indent=2, ensure_ascii=False), encoding='utf-8')

    description = (
        f"Bundled llama-server artifact built on Kaggle.\n"
        f"Profile: {SERVER_PROFILE}.\n"
        f"Dataset slug: {DATASET_USERNAME}/{DATASET_SLUG}.\n"
    )
    metadata = {
        'title': DATASET_TITLE,
        'subtitle': DATASET_SUBTITLE,
        'description': description,
        'id': f'{DATASET_USERNAME}/{DATASET_SLUG}',
        'licenses': [{'name': 'other'}],
    }
    (out_root / "README.generated.md").write_text(description, encoding='utf-8')
    (out_root / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
    return out_root

PACKAGED_DIR = package_runtime_bundle(ACTIVE_BINARY, ACTIVE_BINARY.parent, PACKAGE_ROOT, style=PACKAGE_STYLE, use_cuda_runtime=USE_CUDA_RUNTIME)
print('PACKAGED_DIR =', PACKAGED_DIR)
for item in sorted(PACKAGED_DIR.iterdir()):
    print(' -', item.name)


In [ ]:
# Process 50 code path.
require_process(50)

# This cell keeps the publish decision shallow: build the Kaggle CLI command,
# print the resulting process output, and fail loudly if the upload step fails.


def dataset_publish_command(path: Path, mode: str, note: str, make_public: bool) -> list[str]:
    """Build the Kaggle CLI command for dataset creation or versioning."""
    if mode == 'create':
        cmd = ['kaggle', 'datasets', 'create', '-p', str(path)]
        if make_public:
            cmd.append('--public')
    elif mode == 'version':
        cmd = ['kaggle', 'datasets', 'version', '-p', str(path), '-m', note]
    else:
        raise ValueError(f'Unsupported DATASET_MODE: {mode}')
    if PACKAGE_STYLE == 'runtime_zip':
        cmd += ['--dir-mode', 'zip']
    return cmd


def publish_dataset(path: Path, mode: str, note: str, make_public: bool) -> None:
    """Create or version a Kaggle dataset artifact from the prepared package directory."""
    if mode == 'skip':
        print('Skipping dataset upload')
        return
    proc = subprocess.run(dataset_publish_command(path, mode, note, make_public), text=True, capture_output=True)
    print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode == 0:
        return
    raise RuntimeError(f'Kaggle CLI failed with exit code {proc.returncode}')


RUN_PUBLISH = DATASET_MODE != 'skip'
if guarded_toggle(RUN_PUBLISH, process_id=50, label="Dataset publish"):
    publish_dataset(PACKAGED_DIR, DATASET_MODE, VERSION_NOTE, MAKE_PUBLIC)


## Process 60 — Isolated runtime sandbox and server management

This is the runtime core of the notebook. The helpers in this block prepare a sandbox, construct a predictable execution environment, manage server lifecycle, and capture resource state. They are grouped because they collectively answer one operational question: can this runtime be launched and managed reproducibly on the current Kaggle VM?


In [ ]:
# Process 60 code path.
require_process(60)

def make_runtime_sandbox(source_root: Path, force_new: bool = True) -> Path:
    """Create a throwaway runtime directory so server runs do not mutate the source bundle."""
    existing = sorted([p for p in SANDBOX_BASE.glob("runtime_*") if p.is_dir()])
    if existing and not force_new:
        sandbox_root = existing[-1]
    else:
        sandbox_root = SANDBOX_BASE / f"runtime_{int(time.time())}"
        if sandbox_root.exists():
            shutil.rmtree(sandbox_root)
        sandbox_root.mkdir(parents=True, exist_ok=True)
        for p in source_root.iterdir():
            if p.is_file():
                shutil.copy2(p, sandbox_root / p.name)
    for maybe_exec in [sandbox_root / 'llama-server', sandbox_root / 'llama-server.sh', sandbox_root / 'llama-cli', sandbox_root / 'llama-cli.sh']:
        if maybe_exec.exists():
            maybe_exec.chmod(maybe_exec.stat().st_mode | 0o111)
    return sandbox_root

RUNTIME_SANDBOX = make_runtime_sandbox(ACTIVE_BINARY.parent, force_new=True)
RUNTIME_ROOT = RUNTIME_SANDBOX
sandbox_wrapper = RUNTIME_SANDBOX / 'llama-server.sh'
BINARY_PATH = sandbox_wrapper if sandbox_wrapper.exists() else (RUNTIME_SANDBOX / 'llama-server')
print('RUNTIME_SANDBOX =', RUNTIME_SANDBOX)
print('BINARY_PATH =', BINARY_PATH)


In [ ]:
# Process 60 code path.
require_process(60)

# This block groups runtime lifecycle and monitoring helpers because they share
# the same sandbox, pid, log, and resource-state assumptions. Keeping them together
# reduces hidden coupling between launch, stop, probe, and export code paths.

def clean_env(runtime_root: Path, use_cuda_runtime: bool | None = None) -> dict[str, str]:
    """Build a minimal execution environment for launching llama-server predictably."""
    use_cuda_runtime = USE_CUDA_RUNTIME if use_cuda_runtime is None else use_cuda_runtime
    env = {
        'HOME': os.environ.get('HOME', '/kaggle/working'),
        'PATH': '/usr/bin:/bin',
        'LANG': 'C.UTF-8',
        'LC_ALL': 'C.UTF-8',
    }
    ld_parts = [str(runtime_root)]
    if use_cuda_runtime:
        for part in [
            '/usr/local/nvidia/lib64',
            '/usr/local/cuda/lib64',
            '/usr/local/cuda-12.8/lib64',
            '/usr/local/cuda-12.8/targets/x86_64-linux/lib',
        ]:
            if Path(part).exists():
                ld_parts.append(part)
    if os.environ.get('LD_LIBRARY_PATH'):
        ld_parts.append(os.environ['LD_LIBRARY_PATH'])
    env['LD_LIBRARY_PATH'] = ':'.join(ld_parts)
    return env

def read_pid() -> int | None:
    """Read the stored server pid if one has been recorded for this session."""
    if PID_FILE.exists():
        try:
            return int(PID_FILE.read_text().strip())
        except Exception:
            return None
    return None

def process_state(pid: int) -> str | None:
    """Return the one-letter /proc process state code when available."""
    status_path = Path(f'/proc/{pid}/status')
    if not status_path.exists():
        return None
    for line in status_path.read_text(errors='ignore').splitlines():
        if line.startswith('State:'):
            raw = line.split(':', 1)[1].strip()
            return raw.split()[0] if raw else None
    return None

def process_exists(pid: int) -> bool:
    """Return True when the given pid still exists and is not already a zombie."""
    state = process_state(pid)
    if state == 'Z':
        return False
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

def wait_for_pid_exit(pid: int, timeout_s: float = 20.0, poll_s: float = 0.5) -> bool:
    """Poll until a process exits or the timeout expires."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if not process_exists(pid):
            return True
        time.sleep(poll_s)
    return not process_exists(pid)

def tail_log(n: int = 120) -> str:
    """Return the last log lines for quick notebook-side diagnostics."""
    if not LOG_FILE.exists():
        return ''
    lines = LOG_FILE.read_text(errors='ignore').splitlines()
    return '\n'.join(lines[-n:])

def is_port_open(host: str | None = None, port: int | None = None, timeout: float = 1.0) -> bool:
    """Return True when the configured host and port accept a TCP connection."""
    host = HOST if host is None else host
    port = PORT if port is None else port
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        return s.connect_ex((host, port)) == 0

def _read_meminfo() -> dict[str, int]:
    """Read numeric fields from /proc/meminfo for RAM-based monitoring."""
    data: dict[str, int] = {}
    meminfo = Path('/proc/meminfo')
    if not meminfo.exists():
        return data
    for line in meminfo.read_text().splitlines():
        if ':' not in line:
            continue
        key, raw = line.split(':', 1)
        parts = raw.strip().split()
        if parts and parts[0].isdigit():
            data[key] = int(parts[0])
    return data

def _kb_to_gib(value_kb: int | None) -> float | None:
    """Convert kibibytes to gibibytes with notebook-friendly rounding."""
    if value_kb is None:
        return None
    return round(value_kb / 1024 / 1024, 2)

def system_ram_snapshot() -> dict[str, Any]:
    """Return a coarse RAM snapshot for CPU-profiled sessions."""
    info = _read_meminfo()
    total = info.get('MemTotal')
    available = info.get('MemAvailable')
    used = total - available if total is not None and available is not None else None
    return {
        'monitor_kind': 'ram',
        'mem_total_gib': _kb_to_gib(total),
        'mem_available_gib': _kb_to_gib(available),
        'mem_used_gib': _kb_to_gib(used),
    }

def process_ram_snapshot(pid: int | None = None) -> dict[str, Any]:
    """Return process-local RAM usage for the tracked server pid."""
    pid = pid or read_pid()
    if pid is None:
        return {'pid': None, 'rss_gib': None, 'hwm_gib': None, 'vm_size_gib': None}
    status_path = Path(f'/proc/{pid}/status')
    if not status_path.exists():
        return {'pid': pid, 'rss_gib': None, 'hwm_gib': None, 'vm_size_gib': None}
    raw: dict[str, int] = {}
    for line in status_path.read_text(errors='ignore').splitlines():
        if ':' not in line:
            continue
        key, value = line.split(':', 1)
        parts = value.strip().split()
        if parts and parts[0].isdigit():
            raw[key] = int(parts[0])
    return {
        'pid': pid,
        'rss_gib': _kb_to_gib(raw.get('VmRSS')),
        'hwm_gib': _kb_to_gib(raw.get('VmHWM')),
        'vm_size_gib': _kb_to_gib(raw.get('VmSize')),
    }

def accelerator_snapshot() -> dict[str, Any]:
    """Capture either GPU or RAM state depending on the active server profile."""
    pid = read_pid()
    if MONITOR_KIND == 'gpu':
        cmd = "nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv,noheader,nounits || true"
        proc = subprocess.run(['bash', '-lc', cmd], text=True, capture_output=True)
        rows = []
        for line in (proc.stdout or '').splitlines():
            parts = [part.strip() for part in line.split(',')]
            if len(parts) >= 5:
                rows.append({
                    'index': parts[0],
                    'name': parts[1],
                    'memory_used_mib': parts[2],
                    'memory_total_mib': parts[3],
                    'utilization_gpu_pct': parts[4],
                })
        return {'monitor_kind': 'gpu', 'pid': pid, 'gpus': rows, 'stderr': (proc.stderr or '').strip()}
    return {'monitor_kind': 'ram', 'system': system_ram_snapshot(), 'process': process_ram_snapshot(pid)}

def build_server_command(*, runtime_root: Path, model_file: Path, host: str | None = None, port: int | None = None, ctx_size: int | None = None, batch_size: int | None = None, ubatch_size: int | None = None, parallel: int | None = None, n_gpu_layers: str | int | None = None, flash_attn: str | None = None, split_mode: str | None = None, tensor_split: str | None = None, slots_endpoint: bool | None = None, media_root: Path | None = None, mmproj_file: Path | None = None, extra_args: list[str] | None = None) -> list[str]:
    host = HOST if host is None else host
    port = PORT if port is None else port
    ctx_size = CTX_SIZE if ctx_size is None else ctx_size
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    ubatch_size = UBATCH_SIZE if ubatch_size is None else ubatch_size
    parallel = PARALLEL if parallel is None else parallel
    n_gpu_layers = N_GPU_LAYERS if n_gpu_layers is None else n_gpu_layers
    flash_attn = FLASH_ATTN if flash_attn is None else flash_attn
    split_mode = SPLIT_MODE if split_mode is None else split_mode
    tensor_split = TENSOR_SPLIT if tensor_split is None else tensor_split
    slots_endpoint = SLOTS_ENDPOINT if slots_endpoint is None else slots_endpoint
    media_root = MEDIA_ROOT if media_root is None else media_root
    mmproj_file = MMPROJ_FILE if mmproj_file is None else mmproj_file
    if batch_size is None:
        batch_size = ctx_size
    wrapper = runtime_root / 'llama-server.sh'
    binary = wrapper if wrapper.exists() else (runtime_root / 'llama-server')
    resolved_extra_args = list(EXTRA_SERVER_ARGS if extra_args is None else extra_args)
    cmd = [
        str(binary), '-m', str(model_file), '--host', str(host), '--port', str(port),
        '--ctx-size', str(ctx_size), '--batch-size', str(batch_size), '--ubatch-size', str(ubatch_size),
        '--parallel', str(parallel), '--n-gpu-layers', str(n_gpu_layers), '--flash-attn', str(flash_attn),
    ]
    if split_mode is not None:
        cmd += ['--split-mode', str(split_mode)]
    if tensor_split is not None:
        cmd += ['--tensor-split', str(tensor_split)]
    if slots_endpoint is True:
        cmd.append('--slots')
    elif slots_endpoint is False:
        cmd.append('--no-slots')
    if media_root is not None:
        media_root.mkdir(parents=True, exist_ok=True)
        cmd += ['--media-path', str(media_root)]
    if mmproj_file is not None and Path(mmproj_file).exists():
        cmd += ['--mmproj', str(mmproj_file)]
    cmd.extend(str(x) for x in resolved_extra_args)
    return cmd

def stop_server(force: bool = False, timeout_s: float = 20.0) -> bool:
    pid = read_pid()
    if pid is None:
        PID_FILE.unlink(missing_ok=True)
        print('No PID file found.')
        return True

    state = process_state(pid)
    if state == 'Z':
        PID_FILE.unlink(missing_ok=True)
        print(f'Removed zombie PID file for PID {pid}.')
        return True

    if not process_exists(pid):
        PID_FILE.unlink(missing_ok=True)
        print(f'Removed stale PID file for PID {pid}.')
        return True

    sig = signal.SIGKILL if force else signal.SIGTERM
    os.kill(pid, sig)
    stopped = wait_for_pid_exit(pid, timeout_s=timeout_s)
    if stopped:
        PID_FILE.unlink(missing_ok=True)
        print(f'Stopped PID {pid} with {"SIGKILL" if force else "SIGTERM"}.')
        return True

    if force:
        raise RuntimeError(f'PID {pid} is still alive even after SIGKILL.')
    raise RuntimeError(f'PID {pid} did not exit after SIGTERM. Call stop_server(force=True) only if you explicitly want SIGKILL.')

def wait_for_server(host: str | None = None, port: int | None = None, timeout_s: int | None = None) -> bool:
    import requests
    host = HOST if host is None else host
    port = PORT if port is None else port
    timeout_s = STARTUP_TIMEOUT_SECONDS if timeout_s is None else timeout_s
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if LOG_FILE.exists():
            tail = tail_log(80)
            if 'failed to load model' in tail or 'error loading model' in tail:
                return False
        try:
            r = requests.get(f'http://{host}:{port}/health', timeout=3)
            if r.status_code < 500:
                return True
        except Exception:
            pass
        time.sleep(2)
    return False

def start_server(*, runtime_root: Path | None = None, model_file: Path | None = None, host: str | None = None, port: int | None = None, ctx_size: int | None = None, batch_size: int | None = None, ubatch_size: int | None = None, parallel: int | None = None, n_gpu_layers: str | int | None = None, flash_attn: str | None = None, split_mode: str | None = None, tensor_split: str | None = None, slots_endpoint: bool | None = None, media_root: Path | None = None, mmproj_file: Path | None = None, extra_args: list[str] | None = None, stop_existing: bool = True) -> subprocess.Popen:
    runtime_root = RUNTIME_SANDBOX if runtime_root is None else runtime_root
    model_file = MODEL_FILE if model_file is None else model_file
    host = HOST if host is None else host
    port = PORT if port is None else port
    ctx_size = CTX_SIZE if ctx_size is None else ctx_size
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    ubatch_size = UBATCH_SIZE if ubatch_size is None else ubatch_size
    parallel = PARALLEL if parallel is None else parallel
    n_gpu_layers = N_GPU_LAYERS if n_gpu_layers is None else n_gpu_layers
    flash_attn = FLASH_ATTN if flash_attn is None else flash_attn
    split_mode = SPLIT_MODE if split_mode is None else split_mode
    tensor_split = TENSOR_SPLIT if tensor_split is None else tensor_split
    slots_endpoint = SLOTS_ENDPOINT if slots_endpoint is None else slots_endpoint
    media_root = MEDIA_ROOT if media_root is None else media_root
    mmproj_file = MMPROJ_FILE if mmproj_file is None else mmproj_file
    if stop_existing:
        stop_server(force=False)
    resolved_extra_args = EXTRA_SERVER_ARGS if extra_args is None else extra_args
    cmd = build_server_command(runtime_root=runtime_root, model_file=model_file, host=host, port=port, ctx_size=ctx_size, batch_size=batch_size, ubatch_size=ubatch_size, parallel=parallel, n_gpu_layers=n_gpu_layers, flash_attn=flash_attn, split_mode=split_mode, tensor_split=tensor_split, slots_endpoint=slots_endpoint, media_root=media_root, mmproj_file=mmproj_file, extra_args=resolved_extra_args)
    env = clean_env(runtime_root, use_cuda_runtime=USE_CUDA_RUNTIME)
    with open(LOG_FILE, 'w') as logf:
        proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env)
    PID_FILE.write_text(str(proc.pid), encoding='utf-8')
    CMD_FILE.write_text(json.dumps(cmd, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Started PID:', proc.pid)
    print('Log file:', LOG_FILE)
    print('Monitor:', MONITOR_KIND)
    print('Command:')
    print(' '.join(cmd))
    return proc

def restart_server(**kwargs) -> subprocess.Popen:
    stop_server(force=False)
    time.sleep(2)
    return start_server(**kwargs)


In [ ]:
# Process 60 code path.
require_process(60)

# This cell is intentionally documentation-like rather than executable logic.
# It shows the minimal launch sequence for workflows that use the runtime layer.


def example_runtime_launch_sequence() -> list[str]:
    """Return the recommended manual launch sequence for runtime-oriented sessions."""
    return [
        "proc = start_server(ctx_size=2048, batch_size=2048, ubatch_size=512, parallel=1)",
        "print('Server ready:', wait_for_server(timeout_s=STARTUP_TIMEOUT_SECONDS))",
        "print(tail_log(120))",
    ]


for line in example_runtime_launch_sequence():
    print(line)


## Process 70 — Health, models, and slots

This section probes the running server through lightweight endpoints. The helpers stay together because they describe the same verification surface: whether the launched runtime is alive, loaded, and exposing the expected metadata and slot state.


In [ ]:
# Process 70 code path.
require_process(70)

import requests

# These probes are intentionally lightweight and read-only.
# They let a runtime session confirm server readiness before moving to demos, profiling, or export.

def server_health(base_url: str | None = None):
    """Query the health endpoint and return a notebook-friendly response object."""
    base = base_url or f'http://{HOST}:{PORT}'
    try:
        r = requests.get(f'{base}/health', timeout=5)
        return {'status_code': r.status_code, 'text': r.text}
    except Exception as e:
        return {'error': str(e)}

def server_models(base_url: str | None = None):
    """Return model metadata from either OpenAI-style or legacy model endpoints."""
    base = base_url or f'http://{HOST}:{PORT}'
    for path in ['/v1/models', '/models']:
        try:
            r = requests.get(f'{base}{path}', timeout=10)
            if r.ok:
                return r.json()
        except Exception:
            pass
    return None

def server_slots(base_url: str | None = None):
    """Return slot metadata when the running server exposes it."""
    base = base_url or f'http://{HOST}:{PORT}'
    try:
        r = requests.get(f'{base}/slots', timeout=10)
        return r.json() if r.ok else {'status_code': r.status_code, 'text': r.text}
    except Exception as e:
        return {'error': str(e)}

print('Profile:', SERVER_PROFILE)
print('Port open:', is_port_open())
print('Health:', server_health())
mods = server_models()
print('Models:', json.dumps(mods, indent=2, ensure_ascii=False)[:2000] if mods else None)
slots = server_slots()
print('Slots:', json.dumps(slots, indent=2, ensure_ascii=False)[:2000] if slots else None)
print('Resource snapshot:', json.dumps(accelerator_snapshot(), indent=2, ensure_ascii=False))
print('Log tail:')
print(tail_log(80))


## Process 80 — Text helpers for OpenAI SDK and raw requests

This section contains reusable text-only client helpers for quick smoke tests, JSON-formatting checks, and profiling prompts. They are grouped because they all sit above the same local server contract and differ mainly by transport choice rather than by purpose.


In [ ]:
# Process 80 code path.
require_process(80)

# !pip install -q openai requests httpx

# These text helpers share one purpose: exercise the local server through stable
# chat-style interfaces while keeping transport differences explicit.

import json
import requests
import httpx
from openai import OpenAI

def make_local_client(base_url: str | None = None, api_key: str = 'sk-local', timeout: float | None = None, max_retries: int = 0) -> OpenAI:
    """Build an OpenAI-compatible client for the local llama-server runtime."""
    base_url = f'http://{HOST}:{PORT}/v1' if base_url is None else base_url
    timeout = CLIENT_TIMEOUT_SECONDS if timeout is None else timeout
    return OpenAI(base_url=base_url, api_key=api_key, timeout=httpx.Timeout(timeout, connect=10.0, read=timeout, write=30.0), max_retries=max_retries)

def resolve_model_id(base_url: str | None = None) -> str:
    """Resolve the active model id from the running server, falling back to the GGUF filename."""
    base_url = f'http://{HOST}:{PORT}' if base_url is None else base_url
    data = server_models(base_url)
    if isinstance(data, dict) and data.get('data'):
        return data['data'][0]['id']
    return MODEL_FILE.name

def new_dialog(system: str | None = 'You are a concise and helpful assistant.') -> list[dict]:
    """Create a fresh chat history with an optional system message."""
    msgs = []
    if system:
        msgs.append({'role': 'system', 'content': system})
    return msgs

def append_user(messages: list[dict], content: Any) -> list[dict]:
    """Append one user turn and return the same mutable history for convenience."""
    messages.append({'role': 'user', 'content': content})
    return messages

def append_assistant(messages: list[dict], content: str) -> list[dict]:
    """Append one assistant turn and return the same mutable history for convenience."""
    messages.append({'role': 'assistant', 'content': content})
    return messages

def _field(obj: Any, name: str) -> Any:
    """Read a field from either a dict-like or attribute-based response object."""
    if obj is None:
        return None
    if isinstance(obj, dict):
        return obj.get(name)
    return getattr(obj, name, None)

def _extract_text_like(value: Any) -> str:
    """Normalize text payload fragments from SDK and raw JSON response shapes."""
    if value is None:
        return ''
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict):
                if 'text' in item and isinstance(item['text'], str):
                    parts.append(item['text'])
                elif 'content' in item and isinstance(item['content'], str):
                    parts.append(item['content'])
        return ''.join(parts)
    if isinstance(value, dict):
        if 'text' in value and isinstance(value['text'], str):
            return value['text']
        if 'content' in value and isinstance(value['content'], str):
            return value['content']
        return ''
    return str(value)

def _extract_message_text(message: Any) -> str:
    """Prefer visible content, then reasoning content, from a response message object."""
    for field_name in ('content', 'reasoning_content'):
        text = _extract_text_like(_field(message, field_name))
        if text:
            return text
    return ''

def chat_openai(messages: list[dict], *, client: OpenAI | None = None, model: str | None = None, temperature: float = 0.7, max_tokens: int = 256, stream: bool = False, print_output: bool = True, append_response_to_history: bool = True) -> str:
    """Send a chat request through the OpenAI SDK against the local server."""
    if client is None:
        client = make_local_client()
    if model is None:
        model = resolve_model_id()
    if not stream:
        response = client.chat.completions.create(model=model, messages=messages, temperature=temperature, max_tokens=max_tokens, stream=False)
        text = _extract_message_text(response.choices[0].message)
        if print_output:
            print(text)
        if append_response_to_history:
            append_assistant(messages, text)
        return text
    parts = []
    stream_resp = client.chat.completions.create(model=model, messages=messages, temperature=temperature, max_tokens=max_tokens, stream=True)
    for chunk in stream_resp:
        if not chunk.choices:
            continue
        piece = _extract_message_text(chunk.choices[0].delta)
        if piece:
            parts.append(piece)
            if print_output:
                print(piece, end='', flush=True)
    text = ''.join(parts)
    if print_output:
        print()
    if append_response_to_history:
        append_assistant(messages, text)
    return text

def chat_requests(messages: list[dict], *, base_url: str | None = None, model: str | None = None, temperature: float = 0.7, max_tokens: int = 256, stream: bool = False, timeout: int | None = None, print_output: bool = True, append_response_to_history: bool = True, extra_body: dict | None = None) -> str:
    """Send a chat request with raw HTTP for debugging transport-level behavior."""
    base_url = f'http://{HOST}:{PORT}/v1' if base_url is None else base_url
    timeout = CLIENT_TIMEOUT_SECONDS if timeout is None else timeout
    if model is None:
        model = resolve_model_id(base_url.replace('/v1', ''))
    payload = {'model': model, 'messages': messages, 'temperature': temperature, 'max_tokens': max_tokens, 'stream': stream}
    if extra_body:
        payload.update(extra_body)
    url = f'{base_url}/chat/completions'
    if not stream:
        r = requests.post(url, json=payload, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        text = _extract_message_text(data['choices'][0].get('message', {}))
        if print_output:
            print(text)
        if append_response_to_history:
            append_assistant(messages, text)
        return text
    r = requests.post(url, json=payload, timeout=timeout, stream=True)
    r.raise_for_status()
    parts = []
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith('data: '):
            continue
        payload_str = line[6:]
        if payload_str == '[DONE]':
            break
        obj = json.loads(payload_str)
        if not obj.get('choices'):
            continue
        piece = _extract_message_text(obj['choices'][0].get('delta', {}))
        if piece:
            parts.append(piece)
            if print_output:
                print(piece, end='', flush=True)
    text = ''.join(parts)
    if print_output:
        print()
    if append_response_to_history:
        append_assistant(messages, text)
    return text


In [ ]:
# Process 80 code path.
require_process(80)

# The demo is wrapped in a helper so the cell reads as one short execution choice
# instead of a long notebook-only branch.

RUN_DEMO = False


def run_text_demo() -> None:
    """Exercise the text helpers against the current runtime with one short conversation."""
    print("Process 80 demo: exercising text helpers against the current runtime")
    messages = new_dialog('You are a concise assistant.')
    append_user(messages, 'Write a short sci-fi story on the Moon in 120 words or less.')
    text1 = chat_openai(messages, max_tokens=180, stream=False, print_output=True)
    print("\n--- returned from chat_openai ---")
    print(repr(text1[:300]))

    append_user(messages, 'Continue with two more sentences.')
    text2 = chat_requests(messages, max_tokens=100, stream=True, print_output=True)
    print("\n--- returned from chat_requests ---")
    print(repr(text2[:300]))


if guarded_toggle(RUN_DEMO, process_id=80, label="Text demo"):
    run_text_demo()


## Process 90 — Multimodal helpers

This section is intentionally separate from text chat helpers. Multimodal staging, frame extraction, and content assembly add extra dependencies and extra I/O, so they are easier to reason about when isolated behind their own process number and workflow guard.


In [ ]:
# Process 90 code path.
require_process(90)

# Media staging and content-part helpers live together because they translate
# local files into the specific payload shapes expected by multimodal requests.

def guess_mime(path: Path) -> str:
    """Guess a MIME type for a local media file used in multimodal requests."""
    mime, _ = mimetypes.guess_type(str(path))
    return mime or 'application/octet-stream'

def file_to_data_url(path: Path) -> str:
    """Encode a local file as a data URL for inline multimodal payloads."""
    mime = guess_mime(path)
    data = base64.b64encode(path.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{data}'

def ensure_media_root() -> Path:
    """Create the notebook-local media staging directory when needed."""
    root = WORK_ROOT / 'media'
    root.mkdir(parents=True, exist_ok=True)
    return root

def stage_media_file(path: str | Path, media_root: Path | None = None) -> str:
    """Copy a media file into notebook staging and return a file URL reference."""
    src = Path(path)
    dst_root = media_root or ensure_media_root()
    dst = dst_root / src.name
    shutil.copy2(src, dst)
    return f'file://{dst.name}'

def make_image_part(path: str | Path, mode: str = 'base64') -> dict:
    """Build one OpenAI-style image content part from a local file."""
    p = Path(path)
    url = file_to_data_url(p) if mode == 'base64' else stage_media_file(p)
    return {'type': 'image_url', 'image_url': {'url': url}}

def multimodal_user_content(prompt: str, image_paths: list[str | Path] | None = None, mode: str = 'base64') -> list[dict]:
    """Assemble one multimodal user message from text and zero or more images."""
    content = [{'type': 'text', 'text': prompt}]
    for p in image_paths or []:
        content.append(make_image_part(p, mode=mode))
    return content


In [ ]:
# Process 90 code path.
require_process(90)

# These helpers build on the media staging utilities above and add optional
# higher-level workflows such as frame extraction and multimodal question wrappers.

def ensure_cv2():
    """Return True when OpenCV is importable in the current notebook runtime."""
    try:
        import cv2  # noqa: F401
        return True
    except Exception:
        return False

def extract_video_frames(video_path: str | Path, *, fps_sample: float = 1.0, max_frames: int = 12) -> list[Path]:
    """Extract a bounded set of representative frames for multimodal prompting."""
    if not ensure_cv2():
        raise RuntimeError('OpenCV is not available. Install opencv-python if needed.')
    import cv2
    video_path = Path(video_path)
    out_dir = ensure_media_root() / f'frames_{video_path.stem}'
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Failed to open video: {video_path}')
    native_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    step = max(int(round(native_fps / fps_sample)), 1)
    frames = []
    idx = 0
    saved = 0
    while saved < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            out_path = out_dir / f'frame_{saved:03d}.jpg'
            cv2.imwrite(str(out_path), frame)
            frames.append(out_path)
            saved += 1
        idx += 1
    cap.release()
    return frames

def ask_about_images(messages: list[dict], image_paths: list[str | Path], prompt: str, *, mode: str = 'base64', use_openai: bool = True, max_tokens: int = 256, stream: bool = False) -> str:
    """Ask the model about a set of images using the existing text chat helpers."""
    append_user(messages, multimodal_user_content(prompt, image_paths=image_paths, mode=mode))
    return chat_openai(messages, max_tokens=max_tokens, stream=stream) if use_openai else chat_requests(messages, max_tokens=max_tokens, stream=stream)

def ask_about_video_frames(messages: list[dict], video_path: str | Path, prompt: str, *, fps_sample: float = 1.0, max_frames: int = 12, mode: str = 'base64', use_openai: bool = True, max_tokens: int = 256, stream: bool = False):
    """Extract frames from a video, then reuse the image helper against those frames."""
    frames = extract_video_frames(video_path, fps_sample=fps_sample, max_frames=max_frames)
    text = ask_about_images(messages, frames, prompt, mode=mode, use_openai=use_openai, max_tokens=max_tokens, stream=stream)
    return text, frames

def ask_about_audio_experimental(messages: list[dict], audio_path: str | Path, prompt: str, *, mode: str = 'file', use_openai: bool = True, max_tokens: int = 256, stream: bool = False) -> str:
    """Send an experimental audio payload through the multimodal request path."""
    url = file_to_data_url(Path(audio_path)) if mode == 'base64' else stage_media_file(audio_path)
    content = [{'type': 'text', 'text': prompt}, {'type': 'image_url', 'image_url': {'url': url}}]
    append_user(messages, content)
    return chat_openai(messages, max_tokens=max_tokens, stream=stream) if use_openai else chat_requests(messages, max_tokens=max_tokens, stream=stream)


## Process 100 — Export run artifacts

This final section persists the useful evidence from the session. It is deliberately last because artifact export should summarize a finished workflow rather than participate in runtime setup.


In [ ]:
# Process 100 code path.
require_process(100)

# Export helpers are intentionally collected in one place so the session can
# emit one coherent artifact set at the end instead of scattering files across cells.

RUN_EXPORT = False
EXPORT_NAME = None  # Example: '2026-04-06T18-40-00Z_llama_server_t4x2_profile'


def build_run_manifest() -> dict[str, Any]:
    """Build a compact manifest describing the effective runtime configuration."""
    cmd_payload = None
    if CMD_FILE.exists():
        try:
            cmd_payload = json.loads(CMD_FILE.read_text(encoding='utf-8'))
        except Exception:
            cmd_payload = {'raw': CMD_FILE.read_text(errors='ignore')}
    return {
        'session_workflow': SESSION_WORKFLOW,
        'workflow_processes': list(workflow_processes()),
        'server_profile': SERVER_PROFILE,
        'dataset_slug': DATASET_SLUG,
        'model_file': str(MODEL_FILE),
        'mmproj_file': str(MMPROJ_FILE) if MMPROJ_FILE else None,
        'host': HOST,
        'port': PORT,
        'ctx_size': CTX_SIZE,
        'batch_size': BATCH_SIZE,
        'ubatch_size': UBATCH_SIZE,
        'parallel': PARALLEL,
        'n_gpu_layers': N_GPU_LAYERS,
        'flash_attn': FLASH_ATTN,
        'split_mode': SPLIT_MODE,
        'tensor_split': TENSOR_SPLIT,
        'extra_server_args': EXTRA_SERVER_ARGS,
        'command_payload': cmd_payload,
    }


def build_profile_report() -> dict[str, Any]:
    """Build a lightweight runtime report from current probes and log tail."""
    return {
        'session_workflow': SESSION_WORKFLOW,
        'workflow_processes': list(workflow_processes()),
        'max_stable_context_observed': MAX_STABLE_CONTEXT_OBSERVED,
        'resource_snapshot': accelerator_snapshot(),
        'server_health': server_health(),
        'server_models': server_models(),
        'server_slots': server_slots() if SLOTS_ENDPOINT is not False else None,
        'log_tail': tail_log(120),
    }


def artifact_dataset_example_commands(out_root: Path, dataset_id: str = 'YOUR_KAGGLE_USERNAME/llama-server-runtime-artifacts', version_note: str = 'Add exported run artifacts', make_public: bool = False) -> dict[str, Any]:
    """Return a notebook-friendly example for publishing exported artifacts as a Kaggle dataset."""
    metadata = {
        'title': 'llama-server runtime artifacts',
        'id': dataset_id,
        'licenses': [{'name': 'other'}],
    }
    create_cmd = ['kaggle', 'datasets', 'create', '-p', str(out_root)]
    if make_public:
        create_cmd.append('--public')
    version_cmd = ['kaggle', 'datasets', 'version', '-p', str(out_root), '-m', version_note]
    return {
        'metadata_path': str(out_root / 'dataset-metadata.json'),
        'dataset_metadata': metadata,
        'create_command': create_cmd,
        'version_command': version_cmd,
    }


def export_root(export_name: str | None = None) -> Path:
    """Resolve the output directory for this export operation."""
    export_id = export_name or time.strftime('%Y-%m-%dT%H-%M-%SZ')
    return WORK_ROOT / 'artifacts' / export_id


def export_run_artifacts(export_name: str | None = None) -> Path:
    """Write manifest, profile report, and log files into an export directory."""
    out_root = export_root(export_name)
    out_root.mkdir(parents=True, exist_ok=True)

    (out_root / 'run_manifest.json').write_text(
        json.dumps(build_run_manifest(), indent=2, ensure_ascii=False),
        encoding='utf-8',
    )
    (out_root / 'profile_report.json').write_text(
        json.dumps(build_profile_report(), indent=2, ensure_ascii=False),
        encoding='utf-8',
    )
    if LOG_FILE.exists():
        shutil.copy2(LOG_FILE, out_root / LOG_FILE.name)
    example = artifact_dataset_example_commands(out_root)
    (out_root / 'dataset-metadata.json').write_text(
        json.dumps(example['dataset_metadata'], indent=2, ensure_ascii=False),
        encoding='utf-8',
    )
    print('Process 100 artifacts written to', out_root)
    print('Next step: version this directory into a private Kaggle dataset or download it locally.')
    print('Example metadata path:', example['metadata_path'])
    print('Example dataset id:', example['dataset_metadata']['id'])
    if 'YOUR_KAGGLE_USERNAME' in example['dataset_metadata']['id']:
        print('Warning: replace YOUR_KAGGLE_USERNAME in dataset-metadata.json before running kaggle datasets create.')
    print('Example create command:', ' '.join(example['create_command']))
    print('Example version command:', ' '.join(example['version_command']))
    return out_root


if guarded_toggle(RUN_EXPORT, process_id=100, label="Artifact export"):
    export_run_artifacts(EXPORT_NAME)


## Notes

- This notebook is intentionally multi-purpose. Set `SESSION_WORKFLOW` in the run-guard cell before doing anything else.
- A workflow may include several consecutive numbered processes in one Kaggle session.
- Process numbers are visible on purpose: they make notebook order and workflow scope easier to scan than hidden helper state.
- Use lightweight workflow and process guards to avoid mixing build, publish, demo, profiling, and project handoff actions accidentally.
- If a previously generic cell becomes specialized, move it under the relevant process section instead of broadening its scope silently.
- After a Kaggle VM reboot, rerun the **BOOTSTRAP CELL** and then the server-management cell.
- The runtime and sandbox flow stays anchored to the notebook's own staging layout under `WORK_ROOT`.
- `stop_server()` is PID-file only. No port-wide fallback.
- Text helpers read `reasoning_content` if `content` is empty.
- `EXTRA_SERVER_ARGS` is wired through both `build_server_command()` and `start_server()`.
- `SERVER_PROFILE = "gpu"` targets the CUDA bundle and keeps GPU monitoring enabled.
- `SERVER_PROFILE = "cpu"` targets a separate CPU-only dataset slug and switches monitoring to RAM instead of `nvidia-smi`.
- Do not reuse the CUDA dataset slug for the CPU artifact.
- After a useful run, export `run_manifest.json`, `profile_report.json`, and the server log from the final section.
